# Piano CV Pipeline — Vision-Based Press Detection

## Three-Group Approach

| Group | Keyboard | Hand Landmarks | Purpose |
|-------|----------|----------------|----------|
| **A** | Metadata corners | **Refined JSON skeletons** + filtering | Training teacher (best quality) |
| **B** | **Auto detection** (Canny+Hough) | MediaPipe on raw video | **Deployable (pure CV)** |
| **C** | Metadata corners | MediaPipe on raw video | Ablation (shows calibration gain) |

**Key Insight:** Group A uses refined JSON skeletons with temporal filtering to create **high-quality teacher labels**. These train a CNN that Group B (pure CV) can deploy without any annotations.

---

## Computer Vision Contributions

1. **Learned visual press detection (CNN)** — recognizes finger-key contact from pixel appearance (nail angle, skin deformation), not just geometry
2. **Automatic keyboard localization** — eliminates manual calibration via Canny edge detection + Hough transform with parameter sweeps
3. **Temporal consistency (BiLSTM)** — refines noisy per-frame predictions using motion context
4. **Ablation study** — quantifies: what do we lose by going fully automatic?

---
## 0. Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    REPO_URL  = 'https://github.com/esnylmz/computer-vision.git'
    BRANCH    = 'besn5'
    if not os.path.exists('computer-vision'):
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL], check=True)
    os.chdir('computer-vision')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
    # mediapipe-numpy2 provides mp.solutions (removed in mediapipe 0.10.31+)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'mediapipe-numpy2', 'opencv-python', 'tqdm', 'requests',
                    'scikit-learn', 'scipy'], check=True)
    print('\nColab ready')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    print('Local ready')

import numpy as np
import pandas as pd
import cv2
import json
import warnings
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'NumPy   : {np.__version__}')
print(f'OpenCV  : {cv2.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')

---
## 1. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Change these for full run
# ═══════════════════════════════════════════════════════════════
N              = 3       # videos         (full: 60)
CLIP_DURATION  = 20      # seconds        (full: 120)
FRAME_STEP     = 10      # sample rate    (full: 5)
EPOCHS         = 1       # training       (full: 10+)
SEED           = 42

CACHE_DIR  = '/content/data/cache' if IN_COLAB else './data/cache'
OUTPUT_DIR = '/content/outputs'    if IN_COLAB else './outputs'
MODE       = 'smoke'  # or 'full'

print(f'Mode          : {MODE}')
print(f'N             : {N}')
print(f'CLIP_DURATION : {CLIP_DURATION}s')
print(f'FRAME_STEP    : {FRAME_STEP}')
print(f'EPOCHS        : {EPOCHS}')

---
## 2. STEP 1 — Data & Split

In [ ]:
from src.data import sample_and_split, download_manifest_videos, save_manifest

manifest, dataset = sample_and_split(
    N=N, clip_duration=CLIP_DURATION, frame_step=FRAME_STEP,
    cache_dir=CACHE_DIR, seed=SEED,
)

local_paths = download_manifest_videos(manifest, dataset, verify_fps=True)
manifest['local_video_path'] = manifest['video_id'].map(local_paths)

out_dir = Path(OUTPUT_DIR) / MODE
out_dir.mkdir(parents=True, exist_ok=True)
save_manifest(manifest, str(out_dir / 'manifest.json'))

print(f'\nTrain: {(manifest["split"]=="train").sum()} | '
      f'Test: {(manifest["split"]=="test").sum()}')

---
## 3. STEP 2 — Three-Group Landmark Extraction

Extract landmarks for all three groups:
- **Group A:** Refined JSON skeletons (best quality)
- **Group B:** MediaPipe + auto keyboard (pure CV)
- **Group C:** MediaPipe + metadata corners (ablation)